# Validation of the double-spend attack simulator


This notebook compares the Monte Carlo simulator with the Rosenfeld and Nakamoto approximations, checks the effect of the deficit cap, verifies reproducibility, and benchmarks the exponential and Numba implementations.The project uses the canonical implementations in `theory.py` and `simulator.py`. The imports below are therefore the source of truth for the validation cells in the rest of the notebook.


## Setup and imported reference functions

The notebook imports the canonical implementations from the project modules instead of redefining them locally. This guarantees that every validation step exercises the same code used by the rest of the project.

- **Rosenfeld:** success probability for the race process after `z` confirmations.
- **Nakamoto:** Poisson approximation for the number of blocks mined by the attacker during the `z`-confirmation window.

In [ ]:
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd

from theory import rosenfeld, attacker_success_probability as nakamoto
from simulator import simulate_attack, simulate_attack_exp, simulate_attack_fast

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def estimate_probability(successes, n_trials):
    # Temporary helper kept here until the stats utility is extracted.
    p_hat = successes / n_trials
    standard_error = math.sqrt(p_hat * (1.0 - p_hat) / n_trials)
    return p_hat, standard_error


# Sanity checks for the imported implementation.
assert abs(nakamoto(0.1, 5) - 0.0009136822) < 1e-9
assert abs(nakamoto(0.3, 10) - 0.0416604800) < 1e-9

print("simulate_attack.__module__ =", simulate_attack.__module__)
print("rosenfeld.__module__ =", rosenfeld.__module__)
print("Rosenfeld(0.45, 5):", rosenfeld(0.45, 5))
print("Nakamoto(0.1, 5):", nakamoto(0.1, 5))

simulate_attack.__module__ = simulator
rosenfeld.__module__ = theory
Rosenfeld(0.45, 5): 0.7571581091015617
Nakamoto(0.1, 5): 0.0009136821879277743


## Experiment 2a — z-score table

Experimental grid: `q ∈ {0.1, 0.2, 0.3, 0.4}` and `z ∈ {1, 2, 3, 4, 5}`, with `N = 100,000` runs per cell.

The z-score compares the Monte Carlo estimate with each theoretical value using the binomial standard error implied by the theoretical probability:

$$
z_{model} = \frac{\hat p - p_{model}}{\sqrt{p_{model}(1-p_{model})/N}}.
$$

The generated CSV is saved to `results/validation_zscores.csv`.

In [28]:
N_VALIDATION = 100_000
Q_VALUES = [0.1, 0.2, 0.3, 0.4]
Z_VALUES = [1, 2, 3, 4, 5]
VALIDATION_SEED = 2026

def theoretical_z_score(p_hat: float, p_theory: float, n: int) -> float:
    se = math.sqrt(p_theory * (1.0 - p_theory) / n)
    if se == 0:
        return 0.0 if p_hat == p_theory else math.copysign(math.inf, p_hat - p_theory)
    return (p_hat - p_theory) / se


def run_zscore_experiment(n_trials=N_VALIDATION, seed=VALIDATION_SEED):
    rows = []
    seed_sequence = np.random.SeedSequence(seed)
    child_seeds = iter(seed_sequence.spawn(len(Q_VALUES) * len(Z_VALUES)))

    for q in Q_VALUES:
        for z in Z_VALUES:
            rng = np.random.default_rng(next(child_seeds))
            successes = simulate_attack(q, z, n_trials, rng)
            p_hat = successes / n_trials
            p_rosenfeld = rosenfeld(q, z)
            p_nakamoto = nakamoto(q, z)

            rows.append({
                "q": q,
                "z": z,
                "p_hat": p_hat,
                "rosenfeld": p_rosenfeld,
                "z_rosenfeld": theoretical_z_score(p_hat, p_rosenfeld, n_trials),
                "nakamoto": p_nakamoto,
                "z_nakamoto": theoretical_z_score(p_hat, p_nakamoto, n_trials),
            })

    return pd.DataFrame(rows)


validation_df = run_zscore_experiment()
display(validation_df.round(7))
validation_path = RESULTS_DIR / "validation_zscores.csv"
validation_df.to_csv(validation_path, index=False)
print(f"CSV salvo em: {validation_path}")

,q,z,p_hat,rosenfeld,z_rosenfeld,nakamoto,z_nakamoto
0,0.1,1,0.19917,0.200000,-0.656173,0.204587,-4.246638
1,0.1,2,0.05589,0.056000,-0.151291,0.050978,7.062179
2,0.1,3,0.01659,0.017120,-1.292033,0.013172,9.479618
3,0.1,4,0.00536,0.005456,-0.412118,0.003455,10.264829
4,0.1,5,0.00163,0.001782,-1.138516,0.000914,7.497331
5,0.2,1,0.40129,0.400000,0.832691,0.415899,-9.373349
6,0.2,2,0.20946,0.208000,1.137519,0.203929,4.341374
7,0.2,3,0.11396,0.115840,-1.857647,0.103242,11.139454
8,0.2,4,0.06747,0.066688,0.991219,0.052998,20.428131
9,0.2,5,0.03914,0.039163,-0.037299,0.027415,22.705485


CSV salvo em: results/validation_zscores.csv


### How to read it

- If the simulator matches the Rosenfeld process, about 95% of the values of `|z_rosenfeld|` should stay below 2, and none should exceed 4.
- Large `|z_nakamoto|` values across many cases suggest systematic differences between Nakamoto’s approximation and the simulated process.
- A single z-score does not prove equivalence or disagreement; the full grid should be interpreted jointly.

In [29]:
print("Proporção de |z_rosenfeld| < 2:",
      (validation_df["z_rosenfeld"].abs() < 2).mean())
print("Maior |z_rosenfeld|:",
      validation_df["z_rosenfeld"].abs().max())
print("Quantidade de |z_rosenfeld| > 4:",
      (validation_df["z_rosenfeld"].abs() > 4).sum())
print("Quantidade de |z_nakamoto| > 2:",
      (validation_df["z_nakamoto"].abs() > 2).sum())

Proporção de |z_rosenfeld| < 2: 1.0
Maior |z_rosenfeld|: 1.85764679758176
Quantidade de |z_rosenfeld| > 4: 0
Quantidade de |z_nakamoto| > 2: 19


## Experiment 2b — Sensitivity to `max_deficit`

Parameters: `q = 0.45`, `z = 5`, and `max_deficit ∈ {30, 60, 120}`. Sample size: `N = 500,000`.

The comparison uses the Monte Carlo estimate and its empirical standard error. If the three estimates are compatible within sampling uncertainty, this supports the view that the cap of 60 is sufficient for this scenario.

In [30]:
N_DEFICIT = 500_000
DEFICIT_SEED = 4505
DEFICIT_LIMITS = [30, 60, 120]

def run_deficit_experiment(n_trials=N_DEFICIT, seed=DEFICIT_SEED):
    rows = []
    for max_deficit in DEFICIT_LIMITS:
        rng = np.random.default_rng(seed)
        successes = simulate_attack(
            q=0.45, z=5, n_trials=n_trials, rng=rng,
            max_deficit=max_deficit
        )
        p_hat, se = estimate_probability(successes, n_trials)
        rows.append({
            "max_deficit": max_deficit,
            "successes": successes,
            "N": n_trials,
            "p_hat": p_hat,
            "standard_error": se,
            "rosenfeld": rosenfeld(0.45, 5),
            "difference_from_rosenfeld": p_hat - rosenfeld(0.45, 5),
        })
    return pd.DataFrame(rows)


deficit_df = run_deficit_experiment()
display(deficit_df)

,max_deficit,successes,N,p_hat,standard_error,rosenfeld,difference_from_rosenfeld
0,30,378236,500000,0.756472,0.000607,0.757158,-0.000686
1,60,378640,500000,0.757280,0.000606,0.757158,0.000122
2,120,378647,500000,0.757294,0.000606,0.757158,0.000136


### Note for Section IV

Cap 30 introduces a measurable bias (theoretical ≈ 4.8×10⁻⁴, consistent with the observed drop); cap 60 has a theoretical bias of ≈ 1.2×10⁻⁶, negligible.

## Experiment 2c — Reproducibility across seeds

The same scenario is run five times with different seeds. The estimates should vary only by sampling noise while remaining close to the Rosenfeld reference value.

The requested criterion is to stay within `±2` empirical standard errors of the theoretical value.

In [31]:
REPRO_Q = 0.45
REPRO_Z = 5
REPRO_N = 200_000
REPRO_SEEDS = [11, 29, 47, 83, 101]

def run_reproducibility_experiment():
    p_theory = rosenfeld(REPRO_Q, REPRO_Z)
    rows = []
    for seed in REPRO_SEEDS:
        rng = np.random.default_rng(seed)
        successes = simulate_attack(REPRO_Q, REPRO_Z, REPRO_N, rng)
        p_hat, se = estimate_probability(successes, REPRO_N)
        rows.append({
            "seed": seed,
            "successes": successes,
            "N": REPRO_N,
            "p_hat": p_hat,
            "standard_error": se,
            "rosenfeld": p_theory,
            "z_vs_rosenfeld": (p_hat - p_theory) / se if se else np.nan,
            "within_2_se": abs(p_hat - p_theory) <= 2 * se,
        })
    return pd.DataFrame(rows)


repro_df = run_reproducibility_experiment()
display(repro_df)
print("Todas dentro de ±2 erros-padrão:",
      bool(repro_df["within_2_se"].all()))

,seed,successes,N,p_hat,standard_error,rosenfeld,z_vs_rosenfeld,within_2_se
0,11,151473,200000,0.757365,0.000959,0.757158,0.215838,True
1,29,151572,200000,0.757860,0.000958,0.757158,0.732752,True
2,47,151625,200000,0.758125,0.000958,0.757158,1.009780,True
3,83,151232,200000,0.756160,0.000960,0.757158,-1.039522,True
4,101,151335,200000,0.756675,0.000959,0.757158,-0.503515,True


Todas dentro de ±2 erros-padrão: True


## Experiment 3 — Exponential-time simulation

In this variant, each next block is generated by a race between two independent exponential clocks:

- honest clock with rate `p = 1 - q`
- attacker clock with rate `q`

Because \(P(T_{attack} < T_{honest}) = q/(p+q) = q\), the distributional identity implies that the two simulators should produce compatible estimates.

The function can also accumulate attack duration in block-time units by summing `min(t_honest, t_attack)` at each event.

In [ ]:
print("simulate_attack_exp.__module__ =", simulate_attack_exp.__module__)
print("simulate_attack_exp is imported from simulator.py")

### Comparison of the simulators on the same grid

The table compares the two Monte Carlo estimates. The z-score below uses the standard error of the difference between two independent proportions, with `N` runs in each simulator.

In [33]:
N_EXP = 100_000
EXP_SEED = 31415

def run_exponential_comparison(n_trials=N_EXP, seed=EXP_SEED):
    rows = []
    for q in Q_VALUES:
        for z in Z_VALUES:
            rng_discrete = np.random.default_rng(seed + int(q * 1000) * 10 + z)
            rng_exp = np.random.default_rng(seed + int(q * 1000) * 10 + z + 1)

            s_discrete = simulate_attack(q, z, n_trials, rng_discrete)
            s_exp = simulate_attack_exp(q, z, n_trials, rng_exp)
            p_discrete = s_discrete / n_trials
            p_exp = s_exp / n_trials

            se_diff = math.sqrt(
                p_discrete * (1 - p_discrete) / n_trials
                + p_exp * (1 - p_exp) / n_trials
            )
            z_diff = (p_exp - p_discrete) / se_diff if se_diff else 0.0

            rows.append({
                "q": q,
                "z": z,
                "p_discrete": p_discrete,
                "p_exponential": p_exp,
                "difference": p_exp - p_discrete,
                "z_difference": z_diff,
            })
    return pd.DataFrame(rows)


exp_comparison_df = run_exponential_comparison()
display(exp_comparison_df.round(7))

,q,z,p_discrete,p_exponential,difference,z_difference
0,0.1,1,0.19990,0.19971,-0.00019,-0.106252
1,0.1,2,0.05721,0.05614,-0.00107,-1.034769
2,0.1,3,0.01705,0.01656,-0.00049,-0.852399
3,0.1,4,0.00513,0.00548,0.00035,1.077374
4,0.1,5,0.00175,0.00153,-0.00022,-1.215749
5,0.2,1,0.40220,0.40099,-0.00121,-0.551924
6,0.2,2,0.20788,0.20947,0.00159,0.874924
7,0.2,3,0.11423,0.11504,0.00081,0.568526
8,0.2,4,0.06704,0.06701,-0.00003,-0.026826
9,0.2,5,0.03908,0.03880,-0.00028,-0.323646


### Bonus — Attack duration

The duration returned by the exponential variant is expressed in units of the network’s average block time when the total rate is 1. To express it in minutes, multiply by 10 under the 10-minute block assumption.

Durations are recorded for successful attacks only; failed runs stop at the artificial `max_deficit` cutoff, so their duration has no physical meaning.

In [34]:
DURATION_Q = 0.30
DURATION_Z = 5
DURATION_N = 20_000

duration_rng = np.random.default_rng(202611)
duration_successes, attack_durations = simulate_attack_exp(
    DURATION_Q, DURATION_Z, DURATION_N, duration_rng,
    return_durations=True
)

duration_minutes = attack_durations * 10
print(f"Taxa de sucesso: {duration_successes / DURATION_N:.5f}")
print(f"Duração média: {duration_minutes.mean():.2f} minutos")
print(f"Mediana: {np.median(duration_minutes):.2f} minutos")
print(f"Percentil 95: {np.percentile(duration_minutes, 95):.2f} minutos")

Taxa de sucesso: 0.20570
Duração média: 119.50 minutos
Mediana: 108.44 minutos
Percentil 95: 213.81 minutos


## Experiment 4 — Fast version with Numba

The compiled version preserves the same discrete process as `simulate_attack`. Numba does not accept `numpy.random.Generator` directly in this function, so the implementation uses `np.random.seed(seed)` and `np.random.random()`.

The first call includes compilation overhead; subsequent calls are typically much faster. If Numba is not installed, run `%pip install numba` and restart the kernel.

In [35]:
try:
    from numba import njit
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False
    print("Numba não está instalado. Execute: %pip install numba")

In [36]:
print("simulate_attack_fast.__module__ =", simulate_attack_fast.__module__)
print("simulate_attack_fast is imported from simulator.py")

simulate_attack_fast.__module__ = simulator
simulate_attack_fast is imported from simulator.py


### Validation and performance measurement

The validation compares the success counts with the pure Python version using the same seed. The random streams are not guaranteed to be identical across generators or implementations, so the most appropriate statistical check is to compare proportions within Monte Carlo uncertainty. The cell below also measures runtime, separating the first call (compilation) from later calls.

In [37]:
if NUMBA_AVAILABLE:
    q_test, z_test, n_test, seed_test = 0.3, 5, 200_000, 12345

    start = time.perf_counter()
    fast_successes_first = simulate_attack_fast(
        q_test, z_test, n_test, seed_test
    )
    first_call_seconds = time.perf_counter() - start

    start = time.perf_counter()
    fast_successes_second = simulate_attack_fast(
        q_test, z_test, n_test, seed_test + 1
    )
    compiled_call_seconds = time.perf_counter() - start

    start = time.perf_counter()
    python_successes = simulate_attack(
        q_test, z_test, n_test, np.random.default_rng(seed_test + 2)
    )
    python_seconds = time.perf_counter() - start

    print(f"Numba (primeira chamada, inclui compilação): {first_call_seconds:.3f} s")
    print(f"Numba (já compilado): {compiled_call_seconds:.3f} s")
    print(f"Python puro: {python_seconds:.3f} s")
    print(f"Speedup aproximado (Python / Numba compilado): "
          f"{python_seconds / compiled_call_seconds:.2f}x")

    p_fast = fast_successes_second / n_test
    p_python = python_successes / n_test
    se_diff = math.sqrt(
        p_fast * (1 - p_fast) / n_test
        + p_python * (1 - p_python) / n_test
    )
    z_diff = (p_fast - p_python) / se_diff if se_diff else 0.0
    print(f"p_hat Numba: {p_fast:.6f}")
    print(f"p_hat Python: {p_python:.6f}")
    print(f"z-score da diferença: {z_diff:.3f}")
else:
    print("Pulando benchmark: Numba indisponível.")

Numba (primeira chamada, inclui compilação): 2.073 s
Numba (já compilado): 0.598 s
Python puro: 17.885 s
Speedup aproximado (Python / Numba compilado): 29.90x
p_hat Numba: 0.196500
p_hat Python: 0.199415
z-score da diferença: -2.313


### Running grid 2a with Numba (optional, recommended for larger runs)

Use this cell to repeat the validation grid with the compiled version. It keeps the same columns and saves a separate CSV while preserving the result of the Python run.

In [38]:
if NUMBA_AVAILABLE:
    def run_zscore_experiment_numba(n_trials=N_VALIDATION, seed=VALIDATION_SEED):
        rows = []
        index = 0
        for q in Q_VALUES:
            for z in Z_VALUES:
                successes = simulate_attack_fast(
                    q, z, n_trials, seed + index
                )
                p_hat = successes / n_trials
                p_rosenfeld = rosenfeld(q, z)
                p_nakamoto = nakamoto(q, z)
                rows.append({
                    "q": q,
                    "z": z,
                    "p_hat": p_hat,
                    "rosenfeld": p_rosenfeld,
                    "z_rosenfeld": theoretical_z_score(p_hat, p_rosenfeld, n_trials),
                    "nakamoto": p_nakamoto,
                    "z_nakamoto": theoretical_z_score(p_hat, p_nakamoto, n_trials),
                })
                index += 1
        return pd.DataFrame(rows)

    validation_numba_df = run_zscore_experiment_numba()
    display(validation_numba_df.round(7))
    numba_path = RESULTS_DIR / "validation_zscores_numba.csv"
    validation_numba_df.to_csv(numba_path, index=False)
    print(f"CSV Numba salvo em: {numba_path}")

,q,z,p_hat,rosenfeld,z_rosenfeld,nakamoto,z_nakamoto
0,0.1,1,0.20031,0.200000,0.245077,0.204587,-3.352984
1,0.1,2,0.05496,0.056000,-1.430386,0.050978,5.725110
2,0.1,3,0.01681,0.017120,-0.755718,0.013172,10.089818
3,0.1,4,0.00575,0.005456,1.262113,0.003455,12.366559
4,0.1,5,0.00181,0.001782,0.211147,0.000914,9.381298
5,0.2,1,0.39933,0.400000,-0.432483,0.415899,-10.630878
6,0.2,2,0.20712,0.208000,-0.685628,0.203929,2.504832
7,0.2,3,0.11599,0.115840,0.148217,0.103242,13.249204
8,0.2,4,0.06693,0.066688,0.306745,0.052998,19.665895
9,0.2,5,0.03896,0.039163,-0.330733,0.027415,22.356898


CSV Numba salvo em: results/validation_zscores_numba.csv


## Summary of generated files

- `results/validation_zscores.csv`: main table for Experiment 2a, using pure Python.
- `results/validation_zscores_numba.csv`: repeat of Experiment 2a using Numba, when available.

**Methodological note:** With one pre-mined block, Rosenfeld's overtake condition is a + 1 > h, which is equivalent to a ≥ h, the tie condition without pre-mining. The two models therefore define the same event, and Experiment 2a confirms the equivalence numerically.
